1. Install and Import Required Libraries

In [1]:
!pip install -q transformers datasets evaluate accelerate

In [2]:
!pip install -U transformers datasets accelerate evaluate

In [3]:
!pip uninstall -y hf-xet
!pip install -U transformers huggingface_hub

Found existing installation: hf-xet 1.5.1
Uninstalling hf-xet-1.5.1:
  Successfully uninstalled hf-xet-1.5.1
  Using cached hf_xet-1.5.1-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached hf_xet-1.5.1-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.5 MB)


In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

2. Mount Google Drive and Load the Preprocessed Dataset

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
train = pd.read_csv("/content/drive/MyDrive/Data_Science/models/train_cleaned.csv")

print(train.shape)

train.head()

(159547, 10)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,clean_text,word_count
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,explanation why the edits made under my userna...,47
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,d aww ! he match this background colour i am s...,17
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,hey man i am really not trying to edit war it ...,44
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,more i can not make any real suggestion on imp...,116
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,you sir are my hero any chance you remember wh...,15


3. Prepare Features, Labels, and Split the Dataset

In [7]:
labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

X = train["clean_text"]

y = train[labels]

In [8]:
stratify_labels = (y.sum(axis=1) > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=stratify_labels
)

4. Create Hugging Face Datasets for Training and Testing

In [9]:
train_df = X_train.to_frame()

for col in labels:
    train_df[col] = y_train[col].values

test_df = X_test.to_frame()

for col in labels:
    test_df[col] = y_test[col].values

In [10]:
train_dataset = Dataset.from_pandas(train_df)

test_dataset = Dataset.from_pandas(test_df)

5. Load the DistilBERT Tokenizer and Tokenize the Dataset

In [11]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [12]:
def tokenize(batch):

    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=150
    )

In [13]:
train_dataset = train_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/127637 [00:00<?, ? examples/s]

Map:   0%|          | 0/31910 [00:00<?, ? examples/s]

6. Create Multi-Label Target Vectors

In [14]:
def create_labels(example):

    example["labels"] = [
        float(example["toxic"]),
        float(example["severe_toxic"]),
        float(example["obscene"]),
        float(example["threat"]),
        float(example["insult"]),
        float(example["identity_hate"])
    ]

    return example

train_dataset = train_dataset.map(create_labels)

test_dataset = test_dataset.map(create_labels)

Map:   0%|          | 0/127637 [00:00<?, ? examples/s]

Map:   0%|          | 0/31910 [00:00<?, ? examples/s]

7. Prepare the Dataset and Load the DistilBERT Model

In [15]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [16]:
from transformers import AutoModelForSequenceClassification

labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification",
    label2id=label2id,
    id2label=id2label
)

print(model.config)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "toxic",
    "1": "severe_toxic",
    "2": "obscene",
    "3": "threat",
    "4": "insult",
    "5": "identity_hate"
  },
  "initializer_range": 0.02,
  "label2id": {
    "identity_hate": 5,
    "insult": 4,
    "obscene": 2,
    "severe_toxic": 1,
    "threat": 3,
    "toxic": 0
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "problem_type": "multi_label_classification",
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "vocab_size": 30522
}



8. Configure DistilBERT Training Parameters

In [17]:
training_args = TrainingArguments(

    output_dir="./distilbert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    logging_dir="./logs",

    logging_steps=100,

    report_to="none",

    fp16=torch.cuda.is_available()
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


9. Define Evaluation Metrics

In [18]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    hamming_loss
)

import numpy as np


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))

    preds = (probs >= 0.5).astype(int)

    return {

        "hamming_loss": hamming_loss(labels, preds),

        "micro_precision": precision_score(
            labels,
            preds,
            average="micro",
            zero_division=0
        ),

        "micro_recall": recall_score(
            labels,
            preds,
            average="micro",
            zero_division=0
        ),

        "micro_f1": f1_score(
            labels,
            preds,
            average="micro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            labels,
            probs,
            average="macro"
        )
    }

10. Initialize the Trainer and Fine-Tune the DistilBERT Model

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Hamming Loss,Micro Precision,Micro Recall,Micro F1,Macro F1,Roc Auc,Runtime,Samples Per Second,Steps Per Second
1,0.032740,0.037990,0.015105,0.837124,0.722992,0.775883,0.596383,0.989317,39.269200,812.597000,50.803000
2,0.032129,0.037782,0.015371,0.774211,0.811670,0.792498,0.680777,0.990745,39.749800,802.771000,50.189000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Hamming Loss,Micro Precision,Micro Recall,Micro F1,Macro F1,Roc Auc,Runtime,Samples Per Second,Steps Per Second
1,0.032740,0.037990,0.015105,0.837124,0.722992,0.775883,0.596383,0.989317,39.269200,812.597000,50.803000
2,0.032129,0.037782,0.015371,0.774211,0.811670,0.792498,0.680777,0.990745,39.749800,802.771000,50.189000
3,0.021987,0.041029,0.014760,0.794059,0.799105,0.796574,0.692806,0.989424,39.666400,804.459000,50.294000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=23934, training_loss=0.03502834186404931, metrics={'train_runtime': 1767.3229, 'train_samples_per_second': 216.662, 'train_steps_per_second': 13.543, 'total_flos': 1.48613796074826e+16, 'train_loss': 0.03502834186404931, 'epoch': 3.0})

11. Evaluate the Fine-Tuned DistilBERT Model

In [21]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Hamming Loss,Micro Precision,Micro Recall,Micro F1,Macro F1,Roc Auc,Runtime,Samples Per Second,Steps Per Second
0.021987,0.037782,3,0.015371,0.774211,0.811670,0.792498,0.680777,0.990745,39.846700,800.820000,50.067000


{'eval_loss': 0.03778160735964775,
 'eval_hamming_loss': 0.015371356941397681,
 'eval_micro_precision': 0.7742113238738119,
 'eval_micro_recall': 0.8116695551704217,
 'eval_micro_f1': 0.7924980610590143,
 'eval_macro_f1': 0.6807770688922439,
 'eval_roc_auc': 0.990745372824524,
 'eval_runtime': 39.8467,
 'eval_samples_per_second': 800.82,
 'eval_steps_per_second': 50.067}

12. Generate Classification Report for Each Toxicity Label

In [ ]:
predictions = trainer.predict(test_dataset)

logits = predictions.predictions

labels_true = predictions.label_ids

probs = 1 / (1 + np.exp(-logits))

preds = (probs >= 0.5).astype(int)

In [23]:
from sklearn.metrics import classification_report

for i, label in enumerate(labels):

    print("=" * 70)

    print(label.upper())

    print(
        classification_report(
            labels_true[:, i],
            preds[:, i],
            zero_division=0
        )
    )

TOXIC
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.98     28860
         1.0       0.79      0.87      0.83      3050

    accuracy                           0.96     31910
   macro avg       0.89      0.92      0.90     31910
weighted avg       0.97      0.96      0.97     31910

SEVERE_TOXIC
              precision    recall  f1-score   support

         0.0       0.99      1.00      1.00     31614
         1.0       0.55      0.43      0.48       296

    accuracy                           0.99     31910
   macro avg       0.77      0.71      0.74     31910
weighted avg       0.99      0.99      0.99     31910

OBSCENE
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     30255
         1.0       0.86      0.83      0.84      1655

    accuracy                           0.98     31910
   macro avg       0.92      0.91      0.92     31910
weighted avg       0.98      0.98      0.98    

13. Evaluate the DistilBERT Model Using Custom Decision Thresholds

In [ ]:
predictions = trainer.predict(test_dataset)

logits = predictions.predictions

labels_true = predictions.label_ids

probs = 1 / (1 + np.exp(-logits))

thresholds = {
    "toxic": 0.50,
    "severe_toxic": 0.20,
    "obscene": 0.50,
    "threat": 0.30,
    "insult": 0.50,
    "identity_hate": 0.30
}

preds = np.zeros_like(probs)

for i, label in enumerate(labels):
    preds[:, i] = (probs[:, i] >= thresholds[label]).astype(int)

In [25]:
from sklearn.metrics import classification_report

for i, label in enumerate(labels):

    print("=" * 70)

    print(label.upper())

    print(
        classification_report(
            labels_true[:, i],
            preds[:, i],
            zero_division=0
        )
    )

TOXIC
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.98     28860
         1.0       0.79      0.87      0.83      3050

    accuracy                           0.96     31910
   macro avg       0.89      0.92      0.90     31910
weighted avg       0.97      0.96      0.97     31910

SEVERE_TOXIC
              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99     31614
         1.0       0.35      0.77      0.48       296

    accuracy                           0.98     31910
   macro avg       0.67      0.88      0.74     31910
weighted avg       0.99      0.98      0.99     31910

OBSCENE
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     30255
         1.0       0.86      0.83      0.84      1655

    accuracy                           0.98     31910
   macro avg       0.92      0.91      0.92     31910
weighted avg       0.98      0.98      0.98    

14. Save the Fine-Tuned Model and Perform Sample Predictions

In [ ]:
trainer.save_model("DistilBERT_Model")

tokenizer.save_pretrained("DistilBERT_Model")

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/Data_Science/models/DistilBERT_Model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model and tokenizer saved permanently to Google Drive!")

In [28]:
sample_comments = [
    "You are an idiot.",
    "Have a wonderful day.",
    "I will kill you.",
    "Thank you very much."
]

inputs = tokenizer(
    sample_comments,
    padding=True,
    truncation=True,
    max_length=150,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.sigmoid(outputs.logits).cpu().numpy()

predictions = (probabilities >= 0.5).astype(int)

results = pd.DataFrame(
    predictions,
    columns=labels
)

results.insert(0, "Comment", sample_comments)

results

,Comment,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,You are an idiot.,1,0,1,0,1,0
1,Have a wonderful day.,0,0,0,0,0,0
2,I will kill you.,1,0,0,1,0,0
3,Thank you very much.,0,0,0,0,0,0


In [29]:
import os

SAVE_PATH = "/content/drive/MyDrive/Data_Science/models/DistilBERT_Model"

print(os.path.exists(SAVE_PATH))
print(os.listdir(SAVE_PATH))

True
['config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']
